# Short-range streamflow (`chrtout`)

A **live** pull from the public, unsigned `noaa-nwm-pds` bucket — no
credentials. We fetch one recent `short_range` `channel_rt` file: the
whole-CONUS channel-routing output (~14 MB) holding streamflow for all
~2.7 M NHDPlus reaches at one forecast hour.

We pick a cycle date a few days back so it is reliably published, and a
whole-Earth bbox so the request is a plain whole-file download (a
narrower bbox or `sites=` would need the pyramids `PY-G` reader).

In [ ]:
import datetime as dt
import tempfile
from pathlib import Path

from earthlens import EarthLens

cycle_date = (dt.datetime.now() - dt.timedelta(days=3)).strftime('%Y-%m-%d')
out_dir = Path(tempfile.mkdtemp(prefix='nwm_'))
print('cycle date:', cycle_date)

In [ ]:
nwm = EarthLens(data_source="nwm",
    start=cycle_date, end=cycle_date,
    dataset='chrtout', variables=['streamflow'],
    aoi=[-180, -90, 180, 90],
    configuration='short_range', cycles=[0], steps=[1],
    path=str(out_dir),
)
paths = nwm.download(progress_bar=False)
for p in paths:
    print(p.name, f'{p.stat().st_size / 1e6:.1f} MB')

The downloaded file is the native NWM NetCDF. Decoding `channel_rt`
into a tidy `feature_id × time` streamflow table — and subsetting it to
specific reaches or a bounding box — is a *read*, which is the pyramids
`PY-G` capability (unreleased). Until then earthlens fetches the whole
file and leaves the decode to you / pyramids; it never imports
`xarray`/`zarr` itself.

In [ ]:
assert paths, 'expected one channel_rt file for a recent short_range cycle'
print('OUTPUT_KIND:', nwm.OUTPUT_KIND)  # tabular